# California Housing: regression on real data

This notebook accompanies Exercise 7.3. We compare an affine machine, regularised regression, SVR and an MLP using eight features. Here, no exact function $f_*$ is available: all computed errors remain empirical.

## Outline

1. [Loading and splitting](#donnees-california)
2. [Affine machines and SVR](#classiques-california)
3. [MLP with Flax NNX](#mlp-california)
4. [Comparison and residuals](#residus-california)
5. [Tube and support vectors](#epsilon-california)

In [ ]:
from time import perf_counter

import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

In [ ]:
MODE_RAPIDE = True

if MODE_RAPIDE:
    N_DONNEES = 6_000
    EPOQUES_MLP = 45
else:
    N_DONNEES = None
    EPOQUES_MLP = 90

TAILLE_LOT = 128

<a id="donnees-california"></a>
## 1. Loading and splitting

Load the `California Housing` dataset. The first load requires an internet connection; subsequent loads use the `scikit-learn` cache. Fast mode selects 6,000 observations before forming training, validation and test sets.

Normalise each feature using the mean and standard deviation computed **only on the training set**.

In [ ]:
# Complete this section.

<a id="classiques-california"></a>
## 2. Affine machines and SVR

Fit an affine machine, a regularised affine machine and several Gaussian-kernel SVRs. Choose hyperparameters using only the validation loss.

In [ ]:
def indicateurs(z, prediction):
    residu = prediction - z
    return {
        "RMSE": float(np.sqrt(np.mean(residu**2))),
        "MAE": float(np.mean(np.abs(residu))),
    }


def choisir(candidats):
    return min(candidats, key=lambda r: (r[0], r[1]))

In [ ]:
# Complete this section.

<a id="mlp-california"></a>
## 3. MLP with Flax NNX

Write an MLP with two hidden layers. To stabilise optimisation, also normalise the target using only training data; predictions are then converted back to the original units.

In [ ]:
class MLP(nnx.Module):
    def __init__(self, largeur, *, rngs):
        self.W1 = nnx.Linear(8, largeur, rngs=rngs)
        self.W2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.W3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        x = nnx.relu(self.W1(x))
        x = nnx.relu(self.W2(x))
        return self.W3(x).reshape(-1)


def perte_mlp(machine, x, z):
    return jnp.mean((machine(x) - z) ** 2)


@nnx.jit
def pas_mlp(machine, optimizer, x, z):
    valeur, gradient = nnx.value_and_grad(perte_mlp)(machine, x, z)
    optimizer.update(machine, gradient)
    return valeur


def entrainer_mlp(machine, x, z, *, alpha=1e-3, epoques=EPOQUES_MLP, graine=0):
    optimizer = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    generateur = np.random.default_rng(graine)
    historique = []
    debut = perf_counter()
    for _ in range(epoques):
        permutation = generateur.permutation(z.size)
        pertes = []
        for i in range(0, z.size, TAILLE_LOT):
            indices = permutation[i:i + TAILLE_LOT]
            pertes.append(float(pas_mlp(
                machine, optimizer, jnp.asarray(x[indices]), jnp.asarray(z[indices])
            )))
        historique.append(float(np.mean(pertes)))
    return historique, perf_counter() - debut


def predire_mlp(machine, x):
    prediction_normalisee = np.asarray(machine(jnp.asarray(x)))
    return moyenne_z + ecart_z * prediction_normalisee

In [ ]:
# Complete this section.

<a id="residus-california"></a>
## 4. Comparison and residuals

Compare RMSE and MAE on all three sets. Plot test residuals against predictions. Visible structure in residuals indicates a systematic machine error, or a feature of the data, hidden by the mean alone.

In [ ]:
# Complete this section.

<a id="epsilon-california"></a>
## 5. Tube and support vectors

With $C$ and the kernel fixed, vary $\epsilon$. Compare the support-vector frequency and validation and test RMSE and MAE.

In [ ]:
# Complete this section.

## Summary

Normalisation and the training-validation-test split are part of the mathematical experiment. SVR remains very competitive for this moderately sized dataset; the MLP offers a more flexible function class, but its optimisation introduces further choices. Without an exact function, residual analysis is an essential complement to mean errors.